In [58]:
import kagglehub

# Download datasets
try:    
    path = kagglehub.dataset_download("dhoogla/cicids2017", output_dir='./data')
    print("Datasets successfully downloaded")
except:
    print("Dataset download failed")

Datasets successfully downloaded


In [59]:
# Imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import pyarrow_hotfix
import sklearn
from sklearn.model_selection import train_test_split

print(f"NumPy       : {np.__version__}")
print(f"Pandas      : {pd.__version__}")
print(f"Matplotlib  : {plt.matplotlib.__version__}")
print(f"PyTorch     : {torch.__version__}")
print(f"Scikit-Learn: {sklearn.__version__}")

RANDOM_STATE_SEED = 10

NumPy       : 2.2.6
Pandas      : 2.2.3
Matplotlib  : 3.10.6
PyTorch     : 2.8.0
Scikit-Learn: 1.9.1


This project will focus on detecting and classifying Benign data and DDoS data, therefore we will only be using the corresponding .parquet files.  

In [60]:
pyarrow_hotfix.uninstall()

df_benign = pd.read_parquet('data/Benign-Monday-no-metadata.parquet')
df_ddos = pd.read_parquet('data/DDoS-Friday-no-metadata.parquet')

df_benign

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,4,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,3,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,1,2,0,12,0,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,609,7,4,484,414,233,0,69.14286,111.967896,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
458826,6,18738,1,1,6,6,6,6,6.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
458827,17,60797,2,2,80,156,40,40,40.00000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
458828,17,154,2,2,64,96,32,32,32.00000,0.000000,...,32,0.0,0.0,0,0,0.0,0.0,0,0,Benign
458829,17,155,2,2,80,144,40,40,40.00000,0.000000,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


In [61]:
df_ddos

,Protocol,Flow Duration,Total Fwd Packets,Total Backward Packets,Fwd Packets Length Total,Bwd Packets Length Total,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,Fwd Seg Size Min,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
1,6,109,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
2,6,52,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
3,6,34,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
4,6,3,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
221259,6,61,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
221260,6,72,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
221261,6,75,1,1,6,6,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign
221262,6,48,2,0,12,0,6,6,6.0,0.0,...,20,0.0,0.0,0,0,0.0,0.0,0,0,Benign


The two datasets have identical columns, though the DDoS dataset only has about half as many entries as the benign dataset

In [62]:
print(df_ddos['Label'].value_counts(), "\n")
print(df_benign['Label'].value_counts())

Label
DDoS      128014
Benign     93250
Name: count, dtype: int64 

Label
Benign    458831
Name: count, dtype: int64


We can also see that a large portion of the DDoS dataset is benign data. After combining the datasets, we would get approximately a 1:4,5 DDoS to Benign traffic ratio. This imbalance is notable, though not severe. We could undersample the benign labelled data, though we would lose a significant amount of training data. Oversampling the DDoS labelled records could lead to the model overfitting. Instead, we can keep the slightly imbalanced distribution and instead look at certain metrics aside from accuracy to determine how well our model performs. 

In [63]:
df = pd.concat([df_benign, df_ddos], ignore_index=True)

# Number of unique values per column
df[df.columns[df.nunique() < 10]].nunique()

Protocol                3
Fwd PSH Flags           2
Bwd PSH Flags           1
Fwd URG Flags           1
Bwd URG Flags           1
FIN Flag Count          2
SYN Flag Count          2
RST Flag Count          2
PSH Flag Count          2
ACK Flag Count          2
URG Flag Count          2
CWE Flag Count          1
ECE Flag Count          2
Fwd Avg Bytes/Bulk      1
Fwd Avg Packets/Bulk    1
Fwd Avg Bulk Rate       1
Bwd Avg Bytes/Bulk      1
Bwd Avg Packets/Bulk    1
Bwd Avg Bulk Rate       1
Label                   2
dtype: int64

We combine the two datasets into one. We can see that multiple of the dataset's columns only contain one unique value. This makes these columns irrelevant to our model's training. We can therefore drop these columns from the dataset. The features with 2 unique values are mostly booleans, meaning they can be significant to our model's training process. 

We are now ready to split our dataset into Train, Test and Validation subsets. 

In [ ]:
dropped_columns = df.columns[df.nunique() == 1]
df.drop(columns=dropped_columns, inplace=True)

df_train, df_temp = train_test_split(df, test_size=0.3, stratify=df['Label'], random_state=RANDOM_STATE_SEED)
df_test, df_val = train_test_split(df_temp, test_size=0.5, stratify=df_temp['Label'], random_state=RANDOM_STATE_SEED)

68
